## **0. Settings**

In [47]:
import os, sys
sys.path.append(os.path.abspath('..'))

## **1. Import Libraries**

In [48]:
import ast
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from src.training import dataset, data_loader 

## **2. Get the Data**

In [ ]:
it_ai_human_df = pd.read_csv('../data/preprocessing/it_ai_human_preprocessing_data.csv')
it_ai_human_df.head()

,cleaned_text,cleaned_original_text,tokens,input_ids,attention_mask,generated_type,label
0,Sự xuất hiện của iPhone 17e đánh dấu bước tiếp...,NaN,"{'input_ids': tensor([[ 0, 2470, 621, ...,...","[0, 2470, 621, 447, 275, 4189, 1377, 72, 905, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",human,0
1,"Trong Web3, khái niệm về danh tính đang được x...",NaN,"{'input_ids': tensor([[ 0, 1222, 7572, ...,...","[0, 1222, 7572, 22, 15, 4401, 2036, 425, 1134,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",generated,1
2,**Hướng dẫn chi tiết cách kiểm tra phiên bản M...,Không chỉ riêng mình bạn mà nhiều người dùng k...,"{'input_ids': tensor([[ 0, 12322, 8355, ....","[0, 12322, 8355, 925, 907, 993, 677, 977, 818,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",rewrited,1
3,Một chiến dịch tấn công mạng tinh vi mới đang ...,NaN,"{'input_ids': tensor([[ 0, 1591, 823, ...,...","[0, 1591, 823, 797, 1354, 414, 1270, 1197, 851...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",human,0
4,Để có thể xem lại những bài viết đã đăng trên ...,Để xem lại tin Facebook đã đăng thì bạn cần ph...,"{'input_ids': tensor([[ 0, 2574, 293, ...,...","[0, 2574, 293, 404, 1183, 516, 417, 1115, 1410...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",rewrited,1


## **3. Create Group ID**

In [11]:
# Create group ID to avoid leakaging data
it_ai_human_df['group_key'] = np.where(
    it_ai_human_df['generated_type'] == 'rewrited',
    it_ai_human_df['cleaned_original_text'],
    it_ai_human_df['cleaned_text']
)

mask = (it_ai_human_df['generated_type'] == 'generated')
it_ai_human_df.loc[mask, 'group_key'] = 'generated_' + it_ai_human_df.index[mask].astype(str)

it_ai_human_df['group_id'] = pd.factorize(it_ai_human_df['group_key'])[0]

In [12]:
# Check first 5 rows
it_ai_human_df.sort_values('group_id')[['cleaned_text', 'cleaned_original_text', 'group_id', 'generated_type']][:5]

,cleaned_text,cleaned_original_text,group_id,generated_type
0,Sự xuất hiện của iPhone 17e đánh dấu bước tiếp...,NaN,0,human
2464,Sự xuất hiện của iPhone 17e đánh dấu bước tiếp...,Sự xuất hiện của iPhone 17e đánh dấu bước tiếp...,0,rewrited
1,"Trong Web3, khái niệm về danh tính đang được x...",NaN,1,generated
2,**Hướng dẫn chi tiết cách kiểm tra phiên bản M...,Không chỉ riêng mình bạn mà nhiều người dùng k...,2,rewrited
4007,Không chỉ riêng mình bạn mà nhiều người dùng k...,NaN,2,human


## **4. Split Groups**

In [13]:
# Prepare data
X = it_ai_human_df['tokens']
y = it_ai_human_df['label']
groups = it_ai_human_df['group_id']

In [14]:
SEED = 42

# Split 70 / 15 / 15
gss = GroupShuffleSplit(n_splits=1, train_size=0.7, random_state=SEED)
train_idx, temp_idx = next(gss.split(X, y, groups))

In [15]:
X_temp = X.iloc[temp_idx]
y_temp = y.iloc[temp_idx]
groups_temp = groups.iloc[temp_idx]

In [16]:
gss = GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=SEED)
val_idx_local, test_idx_local = next(gss.split(X_temp, y_temp, groups_temp))

In [17]:
val_idx = temp_idx[val_idx_local]
test_idx = temp_idx[test_idx_local]

In [18]:
# Add new col split
it_ai_human_df['split'] = ''
it_ai_human_df.loc[train_idx, 'split'] = 'train'
it_ai_human_df.loc[val_idx, 'split'] = 'val'
it_ai_human_df.loc[test_idx, 'split'] = 'test'

In [19]:
# Check split value counts
it_ai_human_df['split'].value_counts()

split
train    3502
test      766
val       732
Name: count, dtype: int64

In [20]:
# Check whether data leakaged
it_ai_human_df.groupby('group_id')['split'].nunique().max()

np.int64(1)

## **5. Create Pytorch Dataset**

In [22]:
it_ai_human_df = it_ai_human_df.drop(columns='tokens')

In [32]:
# Convert input_ids and attention_mask to list
it_ai_human_df['input_ids'] = it_ai_human_df['input_ids'].apply(ast.literal_eval)
it_ai_human_df['attention_mask'] = it_ai_human_df['attention_mask'].apply(ast.literal_eval)

In [33]:
# Split dataframe
train_df = it_ai_human_df[it_ai_human_df['split'] == 'train']
val_df = it_ai_human_df[it_ai_human_df['split'] == 'val']
test_df = it_ai_human_df[it_ai_human_df['split'] == 'test']

In [ ]:
# Initialize dataset
train_dataset = dataset.AIDetectionDataset(train_df)
val_dataset = dataset.AIDetectionDataset(val_df)
test_dataset = dataset.AIDetectionDataset(test_df)

In [41]:
# Check length of each dataset
print('Length of train dataset:     ', len(train_dataset))
print('Length of validation dataset: ', len(val_dataset))
print('Length of test dataset:       ', len(test_dataset))

Length of train dataset:      3502
Length of validation dataset:  732
Length of test dataset:        766


In [37]:
# Test returned types
train_dataset[0]

{'input_ids': tensor([   0, 1222, 7572,  ...,    1,    1,    1]),
 'attention_mask': tensor([1, 1, 1,  ..., 0, 0, 0]),
 'labels': tensor(1)}

## **6. Create Data Loader**

In [45]:
train_loader = data_loader.create_dataloader(train_dataset)
val_loader = data_loader.create_dataloader(val_dataset, shuffle=False)
test_loader = data_loader.create_dataloader(test_dataset, shuffle=False)

In [51]:
# Test case
batch = next(iter(train_loader))
# Check shape
print("Input ID shape:          ", batch['input_ids'].shape)
print("Attention masked's shape:", batch['attention_mask'].shape)
print("Label's shape:           ", batch['labels'].shape)

Input ID shape:           torch.Size([16, 1024])
Attention masked's shape: torch.Size([16, 1024])
Label's shape:            torch.Size([16])


In [52]:
# Check keys
print("Batch's keys:", batch.keys())

Batch's keys: dict_keys(['input_ids', 'attention_mask', 'labels'])


## **7. Define Model**